# portopt — Portfolio Optimisation Diagnostic Report

This notebook reads pre-computed JSON results from `generate_report.py`
and produces diagnostic visualisations for both MVO and Black-Litterman portfolios.

**Run via:**
```bash
python notebooks/generate_report.py -d assets.json -p params.json
# or from the C++ CLI:
portopt report -d assets.json -p params.json
```

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print('pandas not available — some tables will be skipped')

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── Locate output directory ───────────────────────────────────────────────────
OUT_DIR = Path(os.environ.get('PORTOPT_OUTPUT_DIR', '.'))
print(f'Reading results from: {OUT_DIR.resolve()}')

In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)

def load_if_exists(name):
    p = OUT_DIR / name
    return load_json(p) if p.exists() else None

manifest     = load_if_exists('manifest.json') or {}
mvo_result   = load_if_exists('mvo_result.json')
mvo_frontier = load_if_exists('mvo_frontier.json')
bl_result    = load_if_exists('bl_result.json')
bl_frontier  = load_if_exists('bl_frontier.json')
bl_model     = load_if_exists('bl_model.json')

print('Loaded files:')
for name, obj in [('manifest', manifest), ('mvo_result', mvo_result),
                   ('mvo_frontier', mvo_frontier), ('bl_result', bl_result),
                   ('bl_frontier', bl_frontier), ('bl_model', bl_model)]:
    print(f'  {name}: {"OK" if obj else "not found"}')

## 1. MVO — Optimal Portfolio Weights

In [ ]:
if mvo_result:
    weights = mvo_result['weights']
    tickers = [w['ticker'] for w in weights]
    values  = [w['weight']  for w in weights]
    m       = mvo_result['metrics']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    ax = axes[0]
    colors = ['#1565C0' if v >= 0 else '#C62828' for v in values]
    bars = ax.bar(tickers, values, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('MVO — Optimal Portfolio Weights', fontweight='bold')
    ax.set_ylabel('Weight')
    ax.set_xticklabels(tickers, rotation=45, ha='right')
    for bar, v in zip(bars, values):
        if abs(v) > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.005 * (1 if v >= 0 else -1),
                    f'{v:.1%}', ha='center', va='bottom', fontsize=9)

    # Pie chart (long positions only)
    ax2 = axes[1]
    pos_t = [t for t, v in zip(tickers, values) if v > 0.005]
    pos_v = [v for v in values if v > 0.005]
    ax2.pie(pos_v, labels=pos_t, autopct='%1.1f%%', startangle=90,
            colors=plt.cm.Blues(np.linspace(0.4, 0.9, len(pos_v))))
    ax2.set_title('MVO — Long-Position Breakdown', fontweight='bold')

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'mvo_weights.png', bbox_inches='tight')
    plt.show()

    print(f"\nPortfolio Metrics:")
    print(f"  Expected Return : {m['expected_return']*100:.2f}%")
    print(f"  Volatility      : {m['volatility']*100:.2f}%")
    print(f"  Sharpe Ratio    : {m['sharpe_ratio']:.3f}")
else:
    print('No MVO result found.')

## 2. Efficient Frontier

In [ ]:
def parse_frontier(data):
    """Extract vols, rets, sharpes from a frontier JSON object."""
    pts = data['frontier']
    vols   = [p['metrics']['volatility']*100     for p in pts]
    rets   = [p['metrics']['expected_return']*100 for p in pts]
    sharpes = [p['metrics']['sharpe_ratio']        for p in pts]
    return vols, rets, sharpes

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Risk-return plot ──────────────────────────────────────────────────────────
ax = axes[0]
if mvo_frontier:
    vols, rets, sharpes = parse_frontier(mvo_frontier)
    ax.plot(vols, rets, '-o', markersize=3, color='#1565C0',
            label='MVO Frontier', linewidth=2)
    best = sharpes.index(max(sharpes))
    ax.scatter(vols[best], rets[best], s=200, marker='*', color='#FF6F00',
               zorder=5, label=f'MVO Max Sharpe ({max(sharpes):.2f})')

    # Annotate individual assets
    if mvo_result:
        for w in mvo_result['weights']:
            pass  # Would need individual asset returns/vols for this

if bl_frontier:
    vols_bl, rets_bl, sharpes_bl = parse_frontier(bl_frontier)
    ax.plot(vols_bl, rets_bl, '--s', markersize=3, color='#AD1457',
            label='BL Frontier', linewidth=2)
    best_bl = sharpes_bl.index(max(sharpes_bl))
    ax.scatter(vols_bl[best_bl], rets_bl[best_bl], s=200, marker='*',
               color='#F57F17', zorder=5,
               label=f'BL Max Sharpe ({max(sharpes_bl):.2f})')

ax.set_xlabel('Volatility (%)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Efficient Frontier', fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)

# ── Sharpe ratio along frontier ───────────────────────────────────────────────
ax2 = axes[1]
if mvo_frontier:
    lambdas = [p['risk_aversion'] for p in mvo_frontier['frontier']]
    ax2.semilogx(lambdas, sharpes, '-o', markersize=3,
                 color='#1565C0', label='MVO', linewidth=2)
if bl_frontier:
    lambdas_bl = [p['risk_aversion'] for p in bl_frontier['frontier']]
    ax2.semilogx(lambdas_bl, sharpes_bl, '--s', markersize=3,
                 color='#AD1457', label='BL', linewidth=2)

ax2.set_xlabel('Risk Aversion (λ, log scale)')
ax2.set_ylabel('Sharpe Ratio')
ax2.set_title('Sharpe Ratio vs. Risk Aversion', fontweight='bold')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / 'efficient_frontier.png', bbox_inches='tight')
plt.show()

## 3. Frontier Weight Heatmap

In [ ]:
def plot_weight_heatmap(frontier_data, title, output_name):
    pts    = frontier_data['frontier']
    tickers = [a['ticker'] for a in frontier_data.get(
        'assets', [{'ticker': f'A{i}'} for i in range(len(pts[0]['weights']))])]
    # weights matrix: rows = frontier points, cols = assets
    W = np.array([[w['weight'] for w in p['weights']] for p in pts])
    lambdas = [p['risk_aversion'] for p in pts]

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(W.T, aspect='auto', cmap='Blues',
                   vmin=0, vmax=1, origin='upper')
    ax.set_xticks(range(len(lambdas)))
    ax.set_xticklabels([f'{l:.1f}' for l in lambdas], rotation=90, fontsize=7)
    ax.set_yticks(range(len(tickers)))
    ax.set_yticklabels(tickers)
    ax.set_xlabel('Risk Aversion (λ)')
    ax.set_title(title, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Portfolio Weight')
    plt.tight_layout()
    plt.savefig(OUT_DIR / output_name, bbox_inches='tight')
    plt.show()

if mvo_frontier:
    plot_weight_heatmap(mvo_frontier, 'MVO — Portfolio Weight Allocation Along Frontier',
                        'mvo_weight_heatmap.png')
if bl_frontier:
    plot_weight_heatmap(bl_frontier, 'BL — Portfolio Weight Allocation Along Frontier',
                        'bl_weight_heatmap.png')

## 4. Black-Litterman: Prior vs. Posterior Returns

In [ ]:
if bl_model:
    assets_bl = bl_model['assets']
    tickers   = [a['ticker']           for a in assets_bl]
    prior     = [a['prior_return']*100  for a in assets_bl]
    posterior = [a['posterior_return']*100 for a in assets_bl]

    x = np.arange(len(tickers))
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar(x - w/2, prior,     w, label='Prior (Equilibrium)',
                color='#90CAF9', edgecolor='black', linewidth=0.5)
    b2 = ax.bar(x + w/2, posterior, w, label='Posterior (Black-Litterman)',
                color='#1565C0', edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(tickers, rotation=45, ha='right')
    ax.set_ylabel('Expected Return (%)')
    ax.set_title('Black-Litterman: Prior vs. Posterior Expected Returns',
                 fontweight='bold')
    ax.legend()
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'bl_prior_posterior.png', bbox_inches='tight')
    plt.show()

    if bl_model.get('views'):
        print('\nViews incorporated:')
        for v in bl_model['views']:
            print(f"  View {v['view_index']}: q={v['expected_return']*100:.2f}%  "
                  f"uncertainty={v['uncertainty']:.4f}")
else:
    print('No BL model output found.')

## 5. MVO vs. BL Portfolio Comparison

In [ ]:
if mvo_result and bl_result:
    mvo_w = {w['ticker']: w['weight'] for w in mvo_result['weights']}
    bl_w  = {w['ticker']: w['weight'] for w in bl_result['weights']}
    tickers = list(mvo_w.keys())

    x = np.arange(len(tickers))
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Side-by-side weights
    ax = axes[0]
    ax.bar(x - 0.2, [mvo_w.get(t, 0) for t in tickers], 0.4,
           label='MVO', color='#1565C0', edgecolor='white')
    ax.bar(x + 0.2, [bl_w.get(t, 0)  for t in tickers], 0.4,
           label='BL',  color='#AD1457', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(tickers, rotation=45, ha='right')
    ax.set_ylabel('Weight')
    ax.set_title('Portfolio Weights: MVO vs. BL', fontweight='bold')
    ax.legend()
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    # Metrics table
    ax2 = axes[1]
    ax2.axis('off')
    mvo_m = mvo_result['metrics']
    bl_m  = bl_result['metrics']
    table_data = [
        ['Metric', 'MVO', 'Black-Litterman'],
        ['Expected Return', f"{mvo_m['expected_return']*100:.2f}%",
                            f"{bl_m['expected_return']*100:.2f}%"],
        ['Volatility',      f"{mvo_m['volatility']*100:.2f}%",
                            f"{bl_m['volatility']*100:.2f}%"],
        ['Sharpe Ratio',    f"{mvo_m['sharpe_ratio']:.3f}",
                            f"{bl_m['sharpe_ratio']:.3f}"],
    ]
    tbl = ax2.table(cellText=table_data[1:], colLabels=table_data[0],
                    loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(12)
    tbl.scale(1.5, 2.0)
    ax2.set_title('Portfolio Metrics Comparison', fontweight='bold', pad=20)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'mvo_vs_bl.png', bbox_inches='tight')
    plt.show()
else:
    print('Need both MVO and BL results for comparison.')

## 6. Risk Decomposition

In [ ]:
def marginal_risk_contribution(weights, cov_matrix):
    """Compute marginal risk contribution (MRC) for each asset."""
    w = np.array(weights)
    Sigma = np.array(cov_matrix)
    port_var = w @ Sigma @ w
    port_vol = np.sqrt(port_var)
    mrc = (Sigma @ w) / port_vol
    rc  = w * mrc  # risk contribution
    rc_pct = rc / port_vol * 100
    return rc_pct

if mvo_result and manifest.get('data_path'):
    try:
        raw_data = load_json(manifest['data_path'])
        cov = np.array(raw_data['covariance'])
        mvo_weights_v = np.array([w['weight'] for w in mvo_result['weights']])
        tickers       = [w['ticker'] for w in mvo_result['weights']]
        rc = marginal_risk_contribution(mvo_weights_v, cov)

        fig, ax = plt.subplots(figsize=(10, 5))
        colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(tickers)))
        ax.bar(tickers, rc, color=colors, edgecolor='white')
        ax.set_ylabel('Risk Contribution (%)')
        ax.set_title('MVO — Asset Risk Contributions', fontweight='bold')
        ax.set_xticklabels(tickers, rotation=45, ha='right')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'risk_contributions.png', bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f'Risk decomposition skipped: {e}')
else:
    print('Risk decomposition requires covariance data in the manifest.')

## 7. Summary

In [ ]:
print('═' * 60)
print(' portopt Diagnostic Report Summary')
print('═' * 60)

if mvo_result:
    m = mvo_result['metrics']
    print(f'\nMVO Optimal Portfolio (λ={manifest.get("risk_aversion", "?")})')
    print(f'  Return   : {m["expected_return"]*100:.2f}%')
    print(f'  Volatility: {m["volatility"]*100:.2f}%')
    print(f'  Sharpe   : {m["sharpe_ratio"]:.3f}')

if bl_result:
    m = bl_result['metrics']
    print(f'\nBlack-Litterman Portfolio')
    print(f'  Return   : {m["expected_return"]*100:.2f}%')
    print(f'  Volatility: {m["volatility"]*100:.2f}%')
    print(f'  Sharpe   : {m["sharpe_ratio"]:.3f}')

print(f'\nGenerated files in: {OUT_DIR.resolve()}')
for f in sorted(OUT_DIR.glob('*.png')):
    print(f'  {f.name}')
print('═' * 60)